In [1]:
import pandas as pd

df = pd.read_csv("twcs/twcs.csv") 
print(df.shape)
df.head()

(2811774, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [2]:
brand_counts = df[df['inbound'] == False]['author_id'].value_counts()
print(brand_counts.head(20))

author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
Name: count, dtype: int64


In [3]:
apple_ids = df[df['author_id'] == 'AppleSupport']['tweet_id'].tolist()
apple_customer_ids = df[df['in_response_to_tweet_id'].isin(apple_ids)]['author_id'].unique()

apple_df = df[(df['author_id'] == 'AppleSupport') | (df['author_id'].isin(apple_customer_ids))]
print(apple_df.shape)
apple_df.head()

(187501, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
193,281,115773,True,Sat Oct 28 23:11:01 +0000 2017,"Please tell me why my @115769 has expired, but...",280,NaN
208,297,115781,True,Thu Nov 02 19:11:14 +0000 2017,@AdobeCare Here is an example of an export bef...,"299,300",296.0
211,301,115781,True,Fri Nov 03 15:01:38 +0000 2017,@AdobeCare Solved this via phone support,NaN,300.0
212,298,115781,True,Tue Oct 31 18:07:49 +0000 2017,@AdobeCare InDesign 13.0 is exporting all PDFs...,"296,302",NaN
351,645,115835,True,Wed Nov 01 08:04:52 +0000 2017,@AmazonHelp That page is useless - doesn’t all...,647,644.0


In [4]:
# Apple's replies
apple_replies = df[df['author_id'] == 'AppleSupport']
customer_tweets = df[df['tweet_id'].isin(apple_replies['in_response_to_tweet_id'])]
apple_df = pd.concat([apple_replies, customer_tweets])
print(apple_df.shape)
apple_df.head()

(213485, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
396,696,AppleSupport,False,Tue Oct 31 22:27:49 +0000 2017,@115854 We're here for you. Which version of t...,697,698.0
398,699,AppleSupport,False,Tue Oct 31 22:36:27 +0000 2017,@115854 Lets take a closer look into this issu...,NaN,697.0
401,701,AppleSupport,False,Tue Oct 31 22:26:49 +0000 2017,@115855 Let's go to DM for the next steps. DM ...,NaN,702.0
403,703,AppleSupport,False,Tue Oct 31 22:09:52 +0000 2017,@115855 Any steps tried since it started last ...,702,704.0
405,705,AppleSupport,False,Tue Oct 31 21:57:00 +0000 2017,@115855 That's great it has iOS 11.1 as we can...,"706,704",707.0


In [5]:
apple_df['text'].str.contains('@AdobeCare|@AmazonHelp|@Uber_Support', na=False).sum()

np.int64(5)

In [6]:
pairs = apple_df[apple_df['author_id'] == 'AppleSupport'].merge(
    apple_df[['tweet_id', 'text']],
    left_on='in_response_to_tweet_id',
    right_on='tweet_id',
    suffixes=('_reply', '_customer')
)

pairs = pairs[['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply']]
print(pairs.shape)
pairs.head()

(106650, 4)


,tweet_id_customer,text_customer,tweet_id_reply,text_reply
0,698,@AppleSupport https://t.co/NV0yucs0lB,696,@115854 We're here for you. Which version of t...
1,697,@AppleSupport The newest update. I️ made sure ...,699,@115854 Lets take a closer look into this issu...
2,702,@AppleSupport Tried resetting my settings .. r...,701,@115855 Let's go to DM for the next steps. DM ...
3,704,@AppleSupport This is what it looks like https...,703,@115855 Any steps tried since it started last ...
4,707,@AppleSupport I️ have an iPhone 7 Plus and yes...,705,@115855 That's great it has iOS 11.1 as we can...


In [7]:
import re

def clean_text(text):
    text = re.sub(r'@\w+', '', str(text))       # remove @mentions
    text = re.sub(r'https?://\S+', '', text)     # remove links
    text = text.strip()
    return text

pairs['text_customer_clean'] = pairs['text_customer'].apply(clean_text)
pairs['text_reply_clean'] = pairs['text_reply'].apply(clean_text)

pairs[['text_customer_clean', 'text_reply_clean']].sample(10)

,text_customer_clean,text_reply_clean
35198,"#podcast #app can’t download, subscribe, sort,...","If you're having issues with the Podcast app, ..."
105642,Just cracked my watch screen just before boar...,Send us a DM and let us know your country. We'...
81032,"Since updating to 11.1.2, my 5S loses almost o...",That's certainly not normal behavior at all. W...
49235,I’m so passed off about this i situation fix it,Here’s what you can do to work around the issu...
31049,"Okay what’s going on? I bought Friday, but it...",You're in the right place for help. To get sta...
5534,probleme de clavier ( tellement lent et decale...,We offer support via Twitter in English. Conta...
65539,Everyone I know with an iPhone (all different ...,Let's take a closer look into this in DM. Send...
31500,"So, the WiFi on my iPhone keeps turning itself...",We'd be happy to look into this with you. Ple...
12842,why does my iPhone lose 50% battery by just be...,Let us know via DM the type of iPhone that you...
23418,Thanks half my games don’t work on iOS 11.0.3.,Games are certainly important to have working....


In [8]:
import random

sample = pairs['text_customer_clean'].sample(100, random_state=42).tolist()
for i, text in enumerate(sample[:30]):
    print(f"{i}: {text}\n")

0: iOS 11 keeps freezing my phone and makes apps unresponsive... fix the bug soon

1: It requested that I enter my IPhone ID or password (I think). It appeared like a notification when lifting the phone &amp; gave me the option to do it later; which I chose.

2: look, i just got this iPhone 8 and i would like to ask y’all why is my phone tripping and dying so fast 😐

3: Fix this  every time me type i this bs come up

4: This is what I get and it does not let me in my email is my Apple ID so it does not let me even read my emails :(

5: Yes, all apps updated aswell. It’s not only apps though, it’s even when I’m  imessaging...

6: why isn’t my low power mode working 🤔🧐help?!...

7: Nearly burnt the house down. Official cable and iPad plug, fire risk

8: LIKE WHAT IS THIS

9: I can’t connect to the app store.

10: Yep

11: Awesome new #iPhone update. Now my phone features 93 minutes of life on a full battery charge. Thanks

12: It happens usually at random times

13: Hey   what’s with the

In [9]:
import random

sample = pairs['text_customer_clean'].sample(100, random_state=42).tolist()
for i, text in enumerate(sample[:30]):
    print(f"{i}: {text}\n")

0: iOS 11 keeps freezing my phone and makes apps unresponsive... fix the bug soon

1: It requested that I enter my IPhone ID or password (I think). It appeared like a notification when lifting the phone &amp; gave me the option to do it later; which I chose.

2: look, i just got this iPhone 8 and i would like to ask y’all why is my phone tripping and dying so fast 😐

3: Fix this  every time me type i this bs come up

4: This is what I get and it does not let me in my email is my Apple ID so it does not let me even read my emails :(

5: Yes, all apps updated aswell. It’s not only apps though, it’s even when I’m  imessaging...

6: why isn’t my low power mode working 🤔🧐help?!...

7: Nearly burnt the house down. Official cable and iPad plug, fire risk

8: LIKE WHAT IS THIS

9: I can’t connect to the app store.

10: Yep

11: Awesome new #iPhone update. Now my phone features 93 minutes of life on a full battery charge. Thanks

12: It happens usually at random times

13: Hey   what’s with the

In [10]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Say hello in one word."}]
)
print(response.choices[0].message.content)

Hello


In [11]:
INTENTS = [
    "bug_performance_issue",
    "account_login_problem",
    "hardware_safety_issue",
    "how_to_info_request",
    "order_purchase_billing",
    "general_complaint"
]

def classify_intent(message):
    prompt = f"""Classify this customer support message into exactly one of these intents:

- bug_performance_issue: device/software bugs, crashes, freezing, lag, battery drain after an update
- account_login_problem: Apple ID, password, login, can't access account
- hardware_safety_issue: physical damage, safety hazard, fire/shock risk
- how_to_info_request: genuine question, "how do I...", "is there a way to..."
- order_purchase_billing: payment failures, orders, refunds, billing
- general_complaint: vague frustration, no clear actionable issue

Message: "{message}"

Respond with ONLY the intent label, nothing else."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

# test on one real example
test_msg = pairs['text_customer_clean'].iloc[0]
print(test_msg)
print("→", classify_intent(test_msg))


→ general_complaint


In [12]:
sample_msgs = pairs['text_customer_clean'].sample(10, random_state=1).tolist()

for msg in sample_msgs:
    if len(msg.strip()) < 5:  # skip near-empty ones
        continue
    intent = classify_intent(msg)
    print(f"MSG: {msg}\n→ {intent}\n")

MSG: My mf phone to be acting up! I’ve never even dropped it!!! I’ve had it maybe a month!!!
→ general_complaint

MSG: my Apple Watch Series 2 sometimes has a sticky Digital Crown, making it hard to turn. Would a repair on this be covered by Apple Care?
→ how_to_info_request

MSG: this “I.T” thing gonna be fixed any time soon....
→ general_complaint

MSG: Why tf does my Do Not Disturb not work?? I’ve had it on for 5 days and niggas still disturbing me
→ bug_performance_issue

MSG: when are you gonna fix the I️ ?
→ bug_performance_issue

MSG: Not what we expect from an #iOs11 #ios11update #ios1103
→ general_complaint

MSG: Your latest update was absolute garbage. My apps are crashing all over the place now, then the phone just reboots. Thanks.
→ bug_performance_issue

MSG: After installing 11.0.2, my iPhone battery is acting like quick sand. Revolutionary features at what cost? Funny!
→ bug_performance_issue

MSG: hey bitch! I have the latest update, and I still have to reset my phone 1

In [13]:
golden_sample = pairs.sample(250, random_state=7).reset_index(drop=True)
golden_sample = golden_sample[golden_sample['text_customer_clean'].str.len() > 15]  # drop fragments/empties
print(golden_sample.shape)
golden_sample.to_csv('golden_sample_raw.csv', index=False)

(239, 6)


In [11]:
golden_sample = pd.read_csv('golden_sample_raw.csv')

golden_sample['predicted_intent'] = golden_sample['text_customer_clean'].apply(classify_intent)

golden_sample['true_intent'] = ''          # you'll fill this (correct if wrong, else copy predicted)
golden_sample['reply_quality'] = ''        # good / partial / bad
golden_sample['decision'] = ''             # auto / escalate
golden_sample['notes'] = ''                # optional, for weird cases

golden_sample.to_csv('golden_sample_to_label.csv', index=False)
print("done")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m257v0d8e1ct5wza4dnzeqnr` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7253, Requested 948. Please try again in 1.5075s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [14]:
import time

def classify_intent(message, retries=3):
    prompt = f"""Classify this customer support message into exactly one of these intents:

- bug_performance_issue: device/software bugs, crashes, freezing, lag, battery drain after an update
- account_login_problem: Apple ID, password, login, can't access account
- hardware_safety_issue: physical damage, safety hazard, fire/shock risk
- how_to_info_request: genuine question, "how do I...", "is there a way to..."
- order_purchase_billing: payment failures, orders, refunds, billing
- general_complaint: vague frustration, no clear actionable issue

Message: "{message}"

Respond with ONLY the intent label, nothing else."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 15 * (attempt + 1)
                print(f"Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    return "ERROR"

In [13]:
import time

predictions = []
for i, msg in enumerate(golden_sample['text_customer_clean']):
    predictions.append(classify_intent(msg))
    if i % 10 == 0:
        print(f"{i}/{len(golden_sample)}")
    time.sleep(1.5)  # stay under the free-tier rate limit

golden_sample['predicted_intent'] = predictions

0/239
10/239
20/239
30/239
40/239
50/239
60/239
70/239
80/239
90/239
100/239
110/239
120/239
130/239
140/239
150/239
160/239
170/239
180/239
190/239
200/239
210/239
220/239
230/239


In [14]:
golden_sample['true_intent'] = ''
golden_sample['reply_quality'] = ''
golden_sample['decision'] = ''
golden_sample['notes'] = ''

golden_sample.to_csv('golden_sample_to_label.csv', index=False)
print("saved")

saved


In [15]:
labeled = pd.read_csv('golden_sample_to_label.csv')
print(labeled.shape)
print(labeled['true_intent'].value_counts())
print(labeled['reply_quality'].value_counts())
print(labeled['decision'].value_counts())

(239, 11)
true_intent
bug_performance_issue     132
general_complaint          48
how_to_info_request        41
account_login_problem       7
hardware_safety_issue       6
order_purchase_billing      3
order_purchase              1
account_login               1
Name: count, dtype: int64
reply_quality
decent     118
good        75
decent      39
bad          7
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


In [15]:
labeled['true_intent'] = labeled['true_intent'].str.strip().str.lower()
labeled['true_intent'] = labeled['true_intent'].replace({
    'order_purchase': 'order_purchase_billing',
    'account_login': 'account_login_problem'
})

labeled['reply_quality'] = labeled['reply_quality'].str.strip().str.lower()
labeled['reply_quality'] = labeled['reply_quality'].replace({'decent': 'partial'})

print(labeled['true_intent'].value_counts())
print(labeled['reply_quality'].value_counts())

true_intent
bug_performance_issue     132
general_complaint          48
how_to_info_request        41
account_login_problem       8
hardware_safety_issue       6
order_purchase_billing      4
Name: count, dtype: int64
reply_quality
partial    157
good        75
bad          7
Name: count, dtype: int64


In [16]:
labeled.to_csv('golden_sample_cleaned.csv', index=False)
print("saved")

saved


In [20]:
labeled['decision'] = labeled['decision'].fillna('escalate')  # only if you're okay defaulting missed ones to escalate

In [18]:
labeled = pd.read_csv('golden_sample_cleaned.csv')
labeled['decision'] = labeled['decision'].str.strip().str.lower()
print(labeled['decision'].value_counts())

accuracy = (labeled['predicted_intent'] == labeled['true_intent']).mean()
print(f"\nClassifier accuracy: {accuracy:.1%}")

decision
escalate    185
auto         54
Name: count, dtype: int64

Classifier accuracy: 92.1%


In [19]:
import chromadb

# set up a local vector database
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="apple_resolutions")

# use the good-quality replies as your grounding knowledge base
good_replies = labeled[labeled['reply_quality'] == 'good'].reset_index(drop=True)

collection.add(
    documents=good_replies['text_customer_clean'].tolist(),
    metadatas=[{"reply": r} for r in good_replies['text_reply_clean'].tolist()],
    ids=[str(i) for i in range(len(good_replies))]
)
print(f"Indexed {len(good_replies)} known-good resolutions")

C:\Users\Aayus\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:34<00:00, 2.39MiB/s]


Indexed 75 known-good resolutions


In [20]:
def looks_like_real_resolution(reply):
    reply = reply.lower()
    # filter out brush-offs and non-answers
    bad_signs = ['dm us', 'send us a dm', 'please dm', 'direct message']
    if any(sign in reply for sign in bad_signs):
        return False
    if len(reply.strip()) < 20:  # too short to be a real resolution
        return False
    return True

candidate_pairs = pairs[pairs['text_reply_clean'].apply(looks_like_real_resolution)]
candidate_pairs = candidate_pairs[candidate_pairs['text_customer_clean'].str.len() > 15]
print(f"Candidate resolutions: {len(candidate_pairs)}")

# take a manageable sample for the vector DB
knowledge_base = candidate_pairs.sample(min(3000, len(candidate_pairs)), random_state=1).reset_index(drop=True)
print(f"Using {len(knowledge_base)} for the vector database")

Candidate resolutions: 70195
Using 3000 for the vector database


In [21]:
# reset the collection so we don't duplicate the old 75
chroma_client.delete_collection(name="apple_resolutions")
collection = chroma_client.create_collection(name="apple_resolutions")

collection.add(
    documents=knowledge_base['text_customer_clean'].tolist(),
    metadatas=[{"reply": r} for r in knowledge_base['text_reply_clean'].tolist()],
    ids=[str(i) for i in range(len(knowledge_base))]
)
print(f"Indexed {collection.count()} resolutions")

Indexed 3000 resolutions


In [22]:
def generate_grounded_reply(customer_message, n_examples=3):
    # retrieve similar past cases
    results = collection.query(query_texts=[customer_message], n_results=n_examples)
    retrieved_replies = [meta['reply'] for meta in results['metadatas'][0]]

    context = "\n\n".join([f"Past case reply: {r}" for r in retrieved_replies])

    prompt = f"""You are an Apple support agent. Here are examples of how similar issues were resolved in the past:

{context}

Now draft a reply to this new customer message, consistent with the style and resolution approach shown above:

"{customer_message}"

Reply:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

# test it
test_msg = labeled['text_customer_clean'].iloc[5]
print("CUSTOMER:", test_msg)
print("\nGENERATED REPLY:", generate_grounded_reply(test_msg))

CUSTOMER: Thanks but as mentioned, it only saves it as an image when this is done

GENERATED REPLY: Thanks for the update. Are you using an iOS device? If so, have you updated to the latest iOS version yet? Let us know in DM and we’ll continue there.


In [23]:
results = collection.query(query_texts=[test_msg], n_results=3)
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"MATCH {i+1}")
    print("Similar past customer msg:", doc)
    print("Its resolution reply:", meta['reply'])
    print()

MATCH 1
Similar past customer msg: i screenshot on facetime and it didn’t save ?
Its resolution reply: Is this happening on an iOS device? If so, have you updated to iOS 11.1 yet? Let us know in DM and we'll continue there.

MATCH 2
Similar past customer msg: any way you can disable having to manually save a picture after you take screenshot?
Its resolution reply: Great question. If you don't want to edit the screenshot, you can swipe left on the thumbnail to dismiss it. It will also automatically save to the Photos app. Check out this link for more information on the new features with screenshots:

MATCH 3
Similar past customer msg: i can’t save pictures by pressing them since i got the new update and this is annoying me
Its resolution reply: We're here to help. Let's take this to DM so we can better assist you.



In [24]:
def looks_like_real_resolution(reply):
    reply = reply.lower()
    bad_signs = ['dm', 'direct message', 'private message']  # broadened
    if any(sign in reply for sign in bad_signs):
        return False
    if len(reply.strip()) < 20:
        return False
    return True

candidate_pairs = pairs[pairs['text_reply_clean'].apply(looks_like_real_resolution)]
candidate_pairs = candidate_pairs[candidate_pairs['text_customer_clean'].str.len() > 15]
print(f"Candidate resolutions: {len(candidate_pairs)}")

knowledge_base = candidate_pairs.sample(min(3000, len(candidate_pairs)), random_state=1).reset_index(drop=True)
print(f"Using {len(knowledge_base)} for the vector database")

chroma_client.delete_collection(name="apple_resolutions")
collection = chroma_client.create_collection(name="apple_resolutions")
collection.add(
    documents=knowledge_base['text_customer_clean'].tolist(),
    metadatas=[{"reply": r} for r in knowledge_base['text_reply_clean'].tolist()],
    ids=[str(i) for i in range(len(knowledge_base))]
)
print(f"Indexed {collection.count()} resolutions")

Candidate resolutions: 47886
Using 3000 for the vector database
Indexed 3000 resolutions


In [26]:
print("GENERATED REPLY:", generate_grounded_reply(test_msg))

GENERATED REPLY: We’d love to help you get the file format you need.  
Could you let us know which iOS version you’re running? That will help us give the most accurate steps.

If you’re trying to save a screenshot (or a Markup‑edited image) as a PDF instead of a JPEG/PNG, you can do it with the built‑in Print feature:

1. Open the screenshot in the Photos app.  
2. Tap the **Share** icon (the square with an arrow).  
3. Choose **Print**.  
4. In the Printer Options screen, pinch out on the preview to open the PDF preview.  
5. Tap the **Share** icon again and select **Save to Files** or **Copy to Books** to keep it as a PDF.

If you want to keep any edits you made in Markup, be sure to tap **Done** in the top‑right corner before you hit Share. That finalizes the edits so they’re included in the PDF.

Let us know if this works or if you’re seeing a different issue, and we’ll walk you through the next steps.


In [27]:
sample_for_replies = labeled.sample(25, random_state=3).reset_index(drop=True)

generated = []
for i, msg in enumerate(sample_for_replies['text_customer_clean']):
    generated.append(generate_grounded_reply(msg))
    print(f"{i+1}/25 done")
    time.sleep(2)

sample_for_replies['generated_reply'] = generated
sample_for_replies.to_csv('generated_replies_sample.csv', index=False)
print("saved")

1/25 done
2/25 done
3/25 done
4/25 done
5/25 done
6/25 done
7/25 done
8/25 done
9/25 done
10/25 done
11/25 done
12/25 done
13/25 done
14/25 done
15/25 done
16/25 done
17/25 done
18/25 done
19/25 done
20/25 done
21/25 done
22/25 done
23/25 done
24/25 done
25/25 done
saved


In [28]:
def judge_reply(customer_message, generated_reply):
    prompt = f"""You are evaluating an AI-generated customer support reply for Apple.

Customer message: "{customer_message}"

Generated reply: "{generated_reply}"

Score this reply on a 1-5 scale for each dimension:
- relevance: does it address what the customer actually asked?
- correctness: is the information technically accurate (as far as you can tell)?
- tone: is it appropriately helpful and professional?
- groundedness: does it sound like a real, specific resolution vs. vague/generic?

Respond ONLY in this exact format, nothing else:
relevance: X
correctness: X
tone: X
groundedness: X
overall: X"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

# test on one
test_row = sample_for_replies.iloc[0]
print(judge_reply(test_row['text_customer_clean'], test_row['generated_reply']))

relevance: 5  
correctness: 5  
tone: 5  
groundedness: 4  
overall: 5


In [29]:
judgments = []
for i, row in sample_for_replies.iterrows():
    result = judge_reply(row['text_customer_clean'], row['generated_reply'])
    judgments.append(result)
    print(f"{i+1}/25 done")
    time.sleep(2)

# parse the text output into actual columns
import re

def parse_judgment(text):
    scores = {}
    for line in text.split('\n'):
        match = re.match(r'(\w+):\s*(\d)', line.strip())
        if match:
            scores[match.group(1)] = int(match.group(2))
    return scores

parsed = [parse_judgment(j) for j in judgments]
sample_for_replies['relevance'] = [p.get('relevance') for p in parsed]
sample_for_replies['correctness'] = [p.get('correctness') for p in parsed]
sample_for_replies['tone'] = [p.get('tone') for p in parsed]
sample_for_replies['groundedness'] = [p.get('groundedness') for p in parsed]
sample_for_replies['overall'] = [p.get('overall') for p in parsed]

sample_for_replies.to_csv('judged_replies.csv', index=False)
print(sample_for_replies[['relevance','correctness','tone','groundedness','overall']].describe())

1/25 done
2/25 done
3/25 done
4/25 done
5/25 done
6/25 done
7/25 done
8/25 done
9/25 done
10/25 done
11/25 done
12/25 done
13/25 done
14/25 done
15/25 done
16/25 done
17/25 done
18/25 done
19/25 done
20/25 done
21/25 done
22/25 done
23/25 done
24/25 done
25/25 done
       relevance  correctness  tone  groundedness  overall
count  25.000000    25.000000  25.0     25.000000    25.00
mean    3.920000     4.560000   4.8      3.960000     4.16
std     1.115049     0.820569   0.5      1.098484     0.80
min     1.000000     2.000000   3.0      1.000000     2.00
25%     3.000000     4.000000   5.0      4.000000     4.00
50%     4.000000     5.000000   5.0      4.000000     4.00
75%     5.000000     5.000000   5.0      5.000000     5.00
max     5.000000     5.000000   5.0      5.000000     5.00


In [30]:
# pick 10 of the 25 to hand-score yourself
to_check = sample_for_replies.sample(10, random_state=5)[['text_customer_clean', 'generated_reply', 'overall']]
to_check.to_csv('judge_check_sample.csv', index=False)
print(to_check)

                                  text_customer_clean  \
19  y’all need to fix my phone bc the update turne...   
18  Probably since Feb.  You guys took me through ...   
2   my phone keeps crashing getting me to put my p...   
10  why does this happen when you try typing the l...   
21  Hopefully I won’t see the “Snapchat-Yellow Scr...   
17  It happens on all GPS centric apps:Apple Maps,...   
24  Hey  - when is the keyboard glitch in my iOS g...   
12  I got a new iPhone and I turned my iCloud phot...   
22  Umm  you lil stinking update didn’t work at al...   
5   Just updated my Music Library and music app is...   

                                      generated_reply  overall  
19  Hi there,\n\nWe’re sorry to hear that the rece...        4  
18  Here’s what you can do to work around the issu...        3  
2   Hi there!  \nI’m sorry you’re having trouble w...        4  
10  Here’s what you can do to work around the issu...        5  
21  Hi there!  \nI’m glad we could help you get

In [35]:
checked = pd.read_csv('judge_check_sample.csv')
checked['human_overall'] = [YOUR_SCORE_1, YOUR_SCORE_2, ..., YOUR_SCORE_10]  # in the same row order as printed above

agreement = (checked['overall'] == checked['human_overall']).mean()
diff = (checked['overall'] - checked['human_overall']).abs().mean()

print(f"Exact agreement: {agreement:.1%}")
print(f"Average score difference: {diff:.2f}")

NameError: name 'YOUR_SCORE_1' is not defined

In [36]:
checked = pd.read_csv('judge_check_sample.csv')
print(checked[['overall', 'human_overall']])

agreement = (checked['overall'] == checked['human_overall']).mean()
diff = (checked['overall'] - checked['human_overall']).abs().mean()

print(f"\nExact agreement: {agreement:.1%}")
print(f"Average score difference: {diff:.2f}")

   overall  human_overall
0        4              3
1        3              4
2        4              3
3        5              3
4        5              3
5        4              4
6        5              3
7        4              4
8        5              4
9        4              5

Exact agreement: 20.0%
Average score difference: 1.10


In [38]:
correlation = checked['overall'].corr(checked['human_overall'], method='spearman')
print(f"Spearman correlation: {correlation:.2f}")

Spearman correlation: -0.46
